Before we do anything else, we need to first clean our table a bit

In [19]:
%pip install emoji

   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/608.4 kB ? eta -:--:--
   ---------------------------------------- 608.4/608.4 kB 1.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [20]:
import pandas as pd
import numpy as np
import emoji
import spacy as sp
import nltk as tk
import textblob as tb
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer

In [12]:
df = pd.read_excel('dataset_tiktok-comments-scraper_2026-04-22_19-27-52-195.xlsx')

In [4]:
df.head()

,searchQuery,username,commentText,likeCount,replyCount,commentLanguage,createdAt
0,https://www.tiktok.com/@collinssema/video/7622...,user4262630805318,very good ❤️,0.0,0.0,en,2026-03-30T11:12:10+00:00
1,https://www.tiktok.com/@collinssema/video/7622...,fifiniimahpro,Asante,0.0,0.0,ht,2026-03-30T11:26:29+00:00
2,https://www.tiktok.com/@collinssema/video/7622...,mubemj,like back,0.0,0.0,en,2026-03-30T11:00:59+00:00
3,https://www.tiktok.com/@collinssema/video/7622...,tashabella497,I like it,0.0,0.0,en,2026-03-30T10:38:11+00:00
4,https://www.tiktok.com/@collinssema/video/7622...,user2583792818581,waaa,0.0,0.0,es,2026-03-30T12:31:51+00:00


In [13]:
df["commentText"] = df["commentText"].replace(r'^\s*$', np.nan, regex=True)
df = df.dropna(subset=["commentText"])

In [6]:
len(df)

11871

Now let's loop through the comments column to make one really long string which we will do tokenization etc on...

In [14]:
comments_str = df["commentText"].str.cat(sep=' ')

Remove stopwords, convert emojis into word, word root form, correct spellings, tokenize, turn into dict of unique word and its frequency and then into df

In [17]:
tokens = word_tokenize(comments_str)
tokens = [x for x in tokens if x not in stopwords.words('english')]

In [21]:
# emojis to text
tokens = [emoji.demojize(token, delimiters=("","")) for token in tokens]

BEFORE CORRECTING TEXT SPELLINGS, LET US FIRST DEAL WITH LANGUAGES... I think we will translate after the ranking. What if i could first classify which language it is that is being spoken and then run a spell checker and translator for the different instances. Maybe after this thing we could build something but for Uganda languages... maybe while doing that Grammarly project thing. Maybe we could extract the words and put them in their various lists... could we use a dict for this and still be able to load with spell checker(or would we have to implement our own that could work with it?). I guess we will combine from SALT and Makerere and of course fitler out unique words. I guess we will ask for Claude's help for code for this. Maybe we could estimate thier age? Maybe use a model trained on classifying female and male usernames

differenct lists of different languages.  use spellchecker python library to correct. stem language word. spellcheck before stem. find a way of loading three at a go. Concat into three in one list? I need to use a combination of the deterministic lists and the API to avoid rate limits on the API and not waste it on the API.
For the best results, use a three-step pipeline:
Deterministic List (The Filter): Use your local dictionary list for "low-hanging fruit" (simple, common words) to save on API costs and speed up processing.
Language Detection API: Use the Sunbird language_id endpoint to verify which language you are dealing with before correcting it.
Sunflower Inference (The Brain): For any word or sentence that fails your local list, send it to the Sunbird Sunflower endpoint with an instruction like: "Rewrite this correctly in [Language]: [Text]

Ganda gemma, swahili gemma and textblob. If its english, textblob, if its swahili, swahili gemma and if it's luganda, ganda gemma.  Language Router: FastText
For processing 10,000 rows locally, FastText is the gold standard. It is a library from Meta that can identify 170+ languages (including English, Swahili, and Luganda) in milliseconds per row.

Don't "Double Process": Don't run the script once for spelling and once for sentiment. It will take twice as long. Do them together in the same function.
Gemma for Sentiment: If you use Ganda Gemma for sentiment, use a very low temperature (0.1). This ensures the model gives you a consistent "Positive/Negative" answer rather than getting creative.
Neutral Zone: Comments in Uganda are often polite but contain complaints. I recommend a wide "Neutral" zone (between -0.1 and 0.1) to avoid mislabeling mild feedback as purely negative.

I think also stemming!

In [ ]:
# correct spellings
#tokens = [str(tb.TextBlob(token).correct()) for token in tokens]

Now let's create a new df but with each row being that of a unique word

In [11]:
uganda_demographics = (pd.Series(comments_str).str.split(' ').explode().drop_duplicates().reset_index(drop=True).to_frame(name="word"))
uganda_demographics.head()

,word
0,very
1,good
2,❤️
3,Asante
4,like


In [12]:
len(uganda_demographics)

3234